In [2]:
from google.transit import gtfs_realtime_pb2
from google.protobuf.json_format import MessageToDict
import pandas as pd
from requests import get
 
# Sample GTFS-R URL from Malaysia's Open API
# URL = 'https://api.data.gov.my/gtfs-realtime/vehicle-position/prasarana?category=rapid-bus-kl'
# URL = 'https://api.data.gov.my/gtfs-realtime/vehicle-position/prasarana?category=rapid-bus-mrtfeeder'
# URL = 'https://api.data.gov.my/gtfs-realtime/vehicle-position/prasarana?category=rapid-rail-kl'
 
# Parse the GTFS Realtime feed
feed = gtfs_realtime_pb2.FeedMessage()
response = get(URL)
feed.ParseFromString(response.content)
 
# Extract and print vehicle position information
vehicle_positions = [MessageToDict(entity.vehicle) for entity in feed.entity]
print(f'Total vehicles: {len(vehicle_positions)}')
df = pd.json_normalize(vehicle_positions)

Total vehicles: 41


In [3]:
df

,timestamp,trip.tripId,trip.startTime,trip.startDate,trip.routeId,position.latitude,position.longitude,position.bearing,position.speed,vehicle.id,vehicle.licensePlate
0,1774278667,weekday_U7508_U750801_6,22:51:04,20260323,U7508,3.088462,101.627975,233.20,65.91,SF2113390,SF2113390
1,1774278684,weekday_T6040_T604002_3,22:42:24,20260323,T6040,3.065367,101.632830,243.00,29.63,WB9172J,WB9172J
2,1774278673,weekday_U2500_U250001_24,22:33:13,20260323,U2500,3.204917,101.732180,136.00,33.34,RAPID423,RAPID423
3,1774278674,weekday_U2020_U202001_11,21:46:14,20260323,U2020,3.251550,101.684500,73.00,14.82,WUU6734,WUU6734
4,1774278682,weekday_U6500_U650002_9,23:00:52,20260323,U6500,3.120083,101.679490,0.00,0.00,WB846U,WB846U
5,1774278667,weekday_U2500_U250001_24,22:52:07,20260323,U2500,3.204917,101.723830,55.00,37.04,RAPID421,RAPID421
6,1774278668,weekday_U8210_U821002_6,23:01:08,20260323,U8210,3.131383,101.681840,39.00,25.93,WVC4047,WVC4047
7,1774278672,weekday_U7510_U751001_10,23:05:42,20260323,U7510,3.128883,101.685840,225.00,27.78,RAPID450,RAPID450
8,1774278663,weekday_U1710_U171002_7,22:41:04,20260323,U1710,3.178300,101.692660,171.00,35.19,WUW8410,WUW8410
9,1774278689,weekday_U6000_U600001_20,23:10:26,20260323,U6000,3.134284,101.695440,100.00,26.11,VGJ1946,VGJ1946


# static API

In [31]:
import requests
import zipfile
import io
import pandas as pd

url = "https://api.data.gov.my/gtfs-static/prasarana?category=rapid-bus-mrtfeeder"

# Download
response = requests.get(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))

# Read all text/csv files into dataframes
dataframes = {}
for name in zip_file.namelist():
    with zip_file.open(name) as f:
        dataframes[name] = pd.read_csv(f)
        
# Access individual dataframes
for name, df in dataframes.items():
    print(f"--- {name} ---")
    print(df.head())

--- agency.txt ---
      agency_name                                         agency_url  \
0  MRT Feeder Bus  https://www.myrapid.com.my/traveling-with-us/h...   

     agency_timezone  agency_phone agency_lang  
0  Asia/Kuala_Lumpur  03-7885-2585          en  
--- calendar.txt ---
   service_id  monday  tuesday  wednesday  thursday  friday  saturday  sunday  \
0    26031302       1        0          0         0       0         1       1   
1    26031301       1        1          1         1       1         0       0   

   start_date  end_date  
0    20260323  20260426  
1    20260324  20260501  
--- calendar_dates.txt ---
   service_id      date  exception_type
0    26031302  20260330               2
1    26031302  20260406               2
2    26031302  20260413               2
3    26031302  20260420               2
--- routes.txt ---
   route_id  agency_id  route_short_name route_long_name  route_type
0  30000143        NaN               NaN            T117           3
1  30000044

In [29]:
dataframes

{'agency.txt':       agency_name                                         agency_url  \
 0  MRT Feeder Bus  https://www.myrapid.com.my/traveling-with-us/h...   
 
      agency_timezone  agency_phone agency_lang  
 0  Asia/Kuala_Lumpur  03-7885-2585          en  ,
 'calendar.txt':    service_id  monday  tuesday  wednesday  thursday  friday  saturday  sunday  \
 0    26031302       1        0          0         0       0         1       1   
 1    26031301       1        1          1         1       1         0       0   
 
    start_date  end_date  
 0    20260323  20260426  
 1    20260324  20260501  ,
 'calendar_dates.txt':    service_id      date  exception_type
 0    26031302  20260330               2
 1    26031302  20260406               2
 2    26031302  20260413               2
 3    26031302  20260420               2,
 'routes.txt':     route_id  agency_id  route_short_name route_long_name  route_type
 0   30000143        NaN               NaN            T117           3
 1   30